# Analisis Exploratorio de Metrobus

In [129]:
import pandas as pd
import seaborn as sns

# Mostrar todas las columnas
pd.set_option('display.max_columns', None)

# Mostrar todo el texto en las celdas sin truncar
pd.set_option('display.max_colwidth', None)

### Fase 1 — Exploración inicial del dataset (EDA)

In [130]:
# Cargamos los archivos CSV en los DataFrames

# Tablas de hechos
df_viajes = pd.read_csv("data/fact_viajes.csv").set_index('viaje_id')
df_incidencias = pd.read_csv("data/fact_incidencias.csv").set_index('incidencia_id')
df_mantenimientos = pd.read_csv("data/fact_mantenimiento.csv").set_index('mantenimiento_id')

# Tablas de dimensión
df_lineas = pd.read_csv("data/dim_linea.csv").set_index('linea_id')
df_vehiculos = pd.read_csv("data/dim_vehiculo.csv").set_index('vehiculo_id')
df_conductores = pd.read_csv("data/dim_conductor.csv").set_index('conductor_id')
df_paradas = pd.read_csv("data/dim_parada.csv").set_index('parada_id')
df_tarifas = pd.read_csv("data/dim_tarifa.csv").set_index('tarifa_id')
df_cocheras = pd.read_csv("data/dim_depot.csv").set_index('depot_id')

In [131]:
# Obtenemos un nuevo DataFrame con los metadatos de todos los datasets
dataframes = {
    "viajes": df_viajes,
    "incidencias": df_incidencias,
    "mantenimiento": df_mantenimientos,
    "lineas": df_lineas,
    "vehiculos": df_vehiculos,
    "conductores": df_conductores,
    "paradas": df_paradas,
    "tarifas": df_tarifas,
    "cocheras": df_cocheras
}

In [132]:
metadatos = {
    'DataFrame': [],
    'Columnas': [],
    'Filas': [],
    'Nulos': [],
    '% Nulos': [],
    'Duplicados': [],
    'Memoria MB': []
}

for name, dataframe in dataframes.items():
    nulos = dataframe.isnull().sum().sum()
    filas = dataframe.shape[0]

    metadatos["DataFrame"].append(name)
    metadatos["Columnas"].append(dataframe.shape[1])
    metadatos["Filas"].append(filas)
    metadatos["Nulos"].append(nulos)
    metadatos["% Nulos"].append(round(((nulos/filas) * 100), 2))
    metadatos["Duplicados"].append(dataframe.duplicated().sum())
    metadatos["Memoria MB"].append(round(dataframe.memory_usage(deep=True).sum() / (1024 **2), 3))

df_estructura_general = pd.DataFrame(metadatos).set_index('DataFrame')
df_estructura_general

,Columnas,Filas,Nulos,% Nulos,Duplicados,Memoria MB
DataFrame,,,,,,
viajes,23,50000,1105,2.21,0,22.212
incidencias,15,4000,0,0.00,0,1.378
mantenimiento,14,876,15,1.71,0,0.317
lineas,6,10,0,0.00,0,0.002
vehiculos,11,45,1,2.22,0,0.010
conductores,9,30,1,3.33,0,0.008
paradas,9,120,2,1.67,0,0.029
tarifas,5,9,0,0.00,0,0.001
cocheras,5,3,0,0.00,0,0.000


### Ficha Individual Para Cada DataFrame

In [133]:
def calcular_nulos(columna):
    return columna.isnull().sum()

def calcular_porcentaje_nulos(columna):
    return round((calcular_nulos(columna)/ columna.size) * 100, 2)

def calcular_valores_unicos(columna):
    return len(columna.unique())

In [134]:
def crear_tabla_metadatos(df):

    return pd.DataFrame(
                data= {
                        'Columna': df.columns,
                        'Tipo': df.dtypes,
                        'Nulos': df.apply(calcular_nulos, axis=0), 
                        '% Nulos': df.apply(calcular_porcentaje_nulos, axis=0),
                        'Unicos': df.apply(calcular_valores_unicos, axis=0)

                    }
                ).set_index('Columna')

## **Viajes**

In [135]:
df_viajes_metadatos = crear_tabla_metadatos(df_viajes)
df_viajes_metadatos

,Tipo,Nulos,% Nulos,Unicos
Columna,,,,
linea_id,int64,0,0.00,10
vehiculo_id,int64,0,0.00,42
conductor_id,int64,0,0.00,28
parada_origen_id,int64,0,0.00,40
parada_destino_id,int64,0,0.00,80
fecha,object,0,0.00,1096
anno,int64,0,0.00,3
mes,int64,0,0.00,12
dia_semana,object,0,0.00,14


In [136]:
df_viajes.head(3)

,linea_id,vehiculo_id,conductor_id,parada_origen_id,parada_destino_id,fecha,anno,mes,dia_semana,es_festivo,franja_horaria,hora_salida_prog,hora_salida_real,hora_llegada_real,retraso_salida_min,duracion_real_min,pasajeros_subidos,ocupacion_pct,km_programados,km_recorridos,viaje_completado,consumo,tarifa_predominante_id
viaje_id,,,,,,,,,,,,,,,,,,,,,,,
1,4,2,2,17,116,2023-08-14,2023,8,Monday,False,Tarde punta,17:50,18:09,19:22,19,73,78.0,0.609,14.1,14.1,True,4.69,4
2,4,3,3,5,48,2024-05-20,2024,5,Monday,False,Manana punta,07:10,07:23,08:30,13,67,115.0,0.885,14.1,14.1,True,5.86,2
3,5,4,4,33,110,2023-08-15,2023,8,Tuesday,False,Valle manana,12:30,12:30,13:05,0,35,47.0,0.362,7.3,7.3,True,3.15,3


In [137]:
df_viajes.describe()

,linea_id,vehiculo_id,conductor_id,parada_origen_id,parada_destino_id,anno,mes,retraso_salida_min,duracion_real_min,pasajeros_subidos,ocupacion_pct,km_programados,km_recorridos,consumo,tarifa_predominante_id
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,49960.000000,50000.000000,50000.000000,50000.000000,48935.000000,50000.000000
mean,5.502600,21.496000,14.498800,20.547780,80.454760,2023.000380,6.518820,3.89218,62.600340,61.283907,0.455362,14.014620,13.910457,7.695104,3.606020
std,2.865665,12.120814,8.077258,11.562176,23.126044,0.816917,3.448467,6.76309,23.849155,36.847517,0.264251,4.949592,4.999930,6.221862,2.160808
min,1.000000,1.000000,1.000000,1.000000,41.000000,2022.000000,1.000000,-99.00000,25.000000,1.000000,0.005000,7.300000,2.200000,0.550000,1.000000
25%,3.000000,11.000000,7.000000,11.000000,60.000000,2022.000000,4.000000,1.00000,44.000000,30.000000,0.227000,9.700000,9.700000,3.600000,2.000000
50%,6.000000,21.000000,14.000000,21.000000,80.500000,2023.000000,7.000000,2.00000,58.000000,56.000000,0.422000,12.600000,12.600000,5.340000,3.000000
75%,8.000000,32.000000,21.000000,31.000000,100.000000,2024.000000,10.000000,5.00000,78.000000,91.000000,0.688000,18.400000,18.400000,9.080000,5.000000
max,10.000000,42.000000,28.000000,40.000000,120.000000,2024.000000,12.000000,50.00000,129.000000,184.000000,0.995000,23.500000,23.500000,32.860000,9.000000


In [138]:
df_viajes.describe(include='object')

,fecha,dia_semana,franja_horaria,hora_salida_prog,hora_salida_real,hora_llegada_real
count,50000,50000,50000,50000,50000,50000
unique,1096,14,6,144,1303,1440
top,2023-01-30,Thursday,Valle manana,10:50,22:00,21:27
freq,66,7259,13667,591,179,76


## **Mantenimiento**

In [139]:
df_mantenimientos_metadatos = crear_tabla_metadatos(df_mantenimientos)
df_mantenimientos_metadatos

,Tipo,Nulos,% Nulos,Unicos
Columna,,,,
vehiculo_id,int64,0,0.00,45
depot_id,int64,0,0.00,3
fecha_entrada,object,0,0.00,567
fecha_salida,object,0,0.00,597
anno,int64,0,0.00,4
mes,int64,0,0.00,12
tipo_mantenimiento,object,0,0.00,10
categoria,object,15,1.71,5
es_correctivo,bool,0,0.00,2


In [140]:
df_mantenimientos.head(3)

,vehiculo_id,depot_id,fecha_entrada,fecha_salida,anno,mes,tipo_mantenimiento,categoria,es_correctivo,dias_fuera_servicio,km_en_revision,coste_eur,proveedor,garantia_meses
mantenimiento_id,,,,,,,,,,,,,,
1,1,1,2024-07-24,2024-07-25,2024,7,Cambio aceite,Preventivo,False,1,278454,218.70,TallerBus Norte,0
2,1,1,2023-02-08,2023-02-09,2023,2,Cambio neumaticos,Preventivo,False,1,240059,1068.93,TallerBus Norte,0
3,1,1,2024-10-28,2024-10-29,2024,10,Cambio neumaticos,Preventivo,False,1,493071,722.19,Taller Oficial Mercedes,0


In [141]:
df_mantenimientos.describe()

,vehiculo_id,depot_id,anno,mes,dias_fuera_servicio,km_en_revision,coste_eur,garantia_meses
count,876.000000,876.000000,876.000000,876.000000,876.000000,876.000000,876.000000,876.000000
mean,22.707763,1.850457,2023.252283,6.276256,1.898402,277372.776256,742.520879,2.075342
std,12.839278,0.782238,0.882364,3.668109,1.963658,156043.953693,924.806659,3.876232
min,1.000000,1.000000,2022.000000,1.000000,0.000000,10222.000000,-4083.030000,0.000000
25%,12.000000,1.000000,2023.000000,3.000000,1.000000,143531.500000,237.712500,0.000000
50%,22.000000,2.000000,2023.000000,6.000000,1.000000,266619.500000,455.085000,0.000000
75%,33.000000,2.000000,2024.000000,9.000000,3.000000,415184.750000,922.100000,3.000000
max,45.000000,3.000000,2025.000000,12.000000,10.000000,549625.000000,4987.720000,12.000000


In [142]:
df_mantenimientos.describe(include='object')

,fecha_entrada,fecha_salida,tipo_mantenimiento,categoria,proveedor
count,876,876,876,861,876
unique,567,597,10,4,5
top,2025-01-01,2025-01-02,Reparacion clima,Preventivo,ManteAuto S.L.
freq,47,14,102,359,194


## **Incidencias**

In [143]:
df_incidencias_metadatos = crear_tabla_metadatos(df_incidencias)
df_incidencias_metadatos

,Tipo,Nulos,% Nulos,Unicos
Columna,,,,
viaje_id,int64,0,0.0,4000
vehiculo_id,int64,0,0.0,42
conductor_id,int64,0,0.0,28
linea_id,int64,0,0.0,10
fecha,object,0,0.0,1067
anno,int64,0,0.0,3
mes,int64,0,0.0,12
hora_incidencia,object,0,0.0,1015
tipo_incidencia,object,0,0.0,12


In [144]:
df_incidencias.head(3)

,viaje_id,vehiculo_id,conductor_id,linea_id,fecha,anno,mes,hora_incidencia,tipo_incidencia,categoria,severidad,requiere_retirada,duracion_resolucion_min,vehiculo_sustituto,coste_estimado_eur
incidencia_id,,,,,,,,,,,,,,,
1,33554,39,11,4,2022-06-01,2022,6,19:11,Huelga parcial,Operacional,Alta,True,60,False,0.00
2,9428,21,21,6,2023-02-16,2023,2,20:42,Fallo electrico,Vehiculo,Alta,True,62,False,575.51
3,200,33,5,8,2022-01-20,2022,1,06:21,Accidente leve,Seguridad,Alta,True,29,True,3318.00


In [145]:
df_incidencias.describe()

,viaje_id,vehiculo_id,conductor_id,linea_id,anno,mes,duracion_resolucion_min,coste_estimado_eur
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.00000,4000.000000
mean,24813.572250,21.342250,14.622250,5.562250,2023.028000,6.546250,28.61075,1198.774348
std,14311.117206,12.140674,8.111967,2.850378,0.812331,3.478561,34.06862,2575.401240
min,5.000000,1.000000,1.000000,1.000000,2022.000000,1.000000,5.00000,0.000000
25%,12580.500000,11.000000,8.000000,3.000000,2022.000000,3.000000,6.00000,0.000000
50%,24807.500000,21.000000,15.000000,6.000000,2023.000000,7.000000,17.00000,183.910000
75%,36905.250000,32.000000,22.000000,8.000000,2024.000000,10.000000,37.00000,1204.757500
max,49989.000000,42.000000,28.000000,10.000000,2024.000000,12.000000,333.00000,14985.580000


In [146]:
df_incidencias.describe(include='object')

,fecha,hora_incidencia,tipo_incidencia,categoria,severidad
count,4000,4000,4000,4000,4000
unique,1067,1015,12,5,4
top,2023-08-03,11:00,Condiciones meteo,Vehiculo,Media
freq,11,20,382,1326,1698


## **Conductores**

In [147]:
df_conductores_metadatos = crear_tabla_metadatos(df_conductores)
df_conductores_metadatos

,Tipo,Nulos,% Nulos,Unicos
Columna,,,,
nombre,object,0,0.00,30
anno_incorporacion,int64,0,0.00,15
antiguedad_anos,float64,1,3.33,15
turno_habitual,object,0,0.00,5
depot_id,int64,0,0.00,3
formacion,object,0,0.00,4
licencia_tipo,object,0,0.00,2
activo,bool,0,0.00,2
ausencias_2024,int64,0,0.00,13


In [148]:
df_conductores.head(3)

,nombre,anno_incorporacion,antiguedad_anos,turno_habitual,depot_id,formacion,licencia_tipo,activo,ausencias_2024
conductor_id,,,,,,,,,
1,Carlos Garcia,2009,15.0,Partido,3,Basica,D,True,14
2,Maria Lopez,2013,11.0,Noche (22-06h),1,Basica,D,True,9
3,Juan Martinez,2008,16.0,Manana (06-14h),2,Completa,D+E,True,6


In [149]:
df_conductores.describe()

,anno_incorporacion,antiguedad_anos,depot_id,ausencias_2024
count,30.000000,29.000000,30.000000,30.000000
mean,2013.500000,10.551724,2.166667,8.633333
std,5.532038,5.622571,0.833908,5.047931
min,2005.000000,1.000000,1.000000,0.000000
25%,2008.000000,5.000000,1.250000,6.250000
50%,2013.000000,11.000000,2.000000,8.500000
75%,2018.750000,16.000000,3.000000,13.500000
max,2023.000000,19.000000,3.000000,15.000000


In [150]:
df_conductores.describe(include='object')

,nombre,turno_habitual,formacion,licencia_tipo
count,30,30,30,30
unique,30,5,4,2
top,Carlos Garcia,Noche (22-06h),Basica + Articulado,D+E
freq,1,8,10,18


## **Cocheras**

In [151]:
df_cocheras_metadatos = crear_tabla_metadatos(df_cocheras)
df_cocheras_metadatos

,Tipo,Nulos,% Nulos,Unicos
Columna,,,,
nombre,object,0,0.0,3
barrio,object,0,0.0,3
latitud,float64,0,0.0,3
longitud,float64,0,0.0,3
capacidad_vehiculos,int64,0,0.0,3


In [152]:
df_cocheras.head(3)

,nombre,barrio,latitud,longitud,capacidad_vehiculos
depot_id,,,,,
1,Cochera Norte,Barrio Norte,40.440,-3.720,20
2,Cochera Central,Centro,40.418,-3.700,15
3,Cochera Sur,Barrio Sur,40.395,-3.695,12


In [153]:
df_cocheras.describe()

,latitud,longitud,capacidad_vehiculos
count,3.000000,3.000000,3.000000
mean,40.417667,-3.705000,15.666667
std,0.022502,0.013229,4.041452
min,40.395000,-3.720000,12.000000
25%,40.406500,-3.710000,13.500000
50%,40.418000,-3.700000,15.000000
75%,40.429000,-3.697500,17.500000
max,40.440000,-3.695000,20.000000


In [154]:
df_cocheras.describe(include='object')

,nombre,barrio
count,3,3
unique,3,3
top,Cochera Norte,Barrio Norte
freq,1,1


## **Lineas**

In [155]:
df_lineas_metadatos = crear_tabla_metadatos(df_lineas)
df_lineas_metadatos

,Tipo,Nulos,% Nulos,Unicos
Columna,,,,
codigo,object,0,0.0,10
nombre,object,0,0.0,10
tipo,object,0,0.0,3
km_recorrido,float64,0,0.0,10
n_paradas,int64,0,0.0,10
frecuencia_min,int64,0,0.0,7


In [156]:
df_lineas.head(3)

,codigo,nombre,tipo,km_recorrido,n_paradas,frecuencia_min
linea_id,,,,,,
1,L1,Centro - Aeropuerto,Urbana,18.4,32,12
2,L2,Universidad - Hospital,Urbana,11.2,21,8
3,L3,Barrio Norte - Estacion,Urbana,9.7,18,10


In [157]:
df_lineas.describe()

,km_recorrido,n_paradas,frecuencia_min
count,10.000000,10.000000,10.000000
mean,14.050000,18.900000,17.400000
std,5.200694,6.436873,13.040109
min,7.300000,10.000000,6.000000
25%,10.075000,14.500000,8.500000
50%,13.350000,19.000000,11.000000
75%,17.550000,21.750000,26.250000
max,23.500000,32.000000,45.000000


In [158]:
df_lineas.describe(include='object')

,codigo,nombre,tipo
count,10,10,10
unique,10,10,3
top,L1,Centro - Aeropuerto,Urbana
freq,1,1,7


## **Paradas**

In [159]:
df_paradas_metadatos = crear_tabla_metadatos(df_paradas)
df_paradas_metadatos

,Tipo,Nulos,% Nulos,Unicos
Columna,,,,
nombre_parada,object,0,0.00,120
barrio,object,0,0.00,13
tipo,object,0,0.00,3
latitud,float64,1,0.83,120
longitud,float64,0,0.00,120
accesible_silla,object,1,0.83,3
marquesina,bool,0,0.00,2
panel_informacion,bool,0,0.00,2
activa,bool,0,0.00,2


In [160]:
df_paradas.head(3)

,nombre_parada,barrio,tipo,latitud,longitud,accesible_silla,marquesina,panel_informacion,activa
parada_id,,,,,,,,,
1,Parada Barrio Norte 1,Barrio Norte,Intermedia,40.431989,-3.699183,True,False,False,True
2,Parada Barrio Sur 2,Barrio Sur,Intermedia,40.403568,-3.692632,True,True,True,True
3,Parada Universidad 3,Universidad,Intermedia,40.419744,-3.723256,True,False,False,True


In [161]:
df_paradas.describe()

,latitud,longitud
count,119.000000,120.000000
mean,48.473379,-3.701965
std,87.873063,0.020579
min,40.387789,-3.751602
25%,40.408617,-3.713621
50%,40.418548,-3.700251
75%,40.428715,-3.688889
max,999.000000,-3.658867


In [162]:
df_paradas.describe(include='object')

,nombre_parada,barrio,tipo,accesible_silla
count,120,120,120,119
unique,120,13,3,2
top,Parada Barrio Norte 1,Barrio Norte,Intermedia,True
freq,1,11,108,102


## **Tarifas**

In [163]:
df_tarifas_metadatos = crear_tabla_metadatos(df_tarifas)
df_tarifas_metadatos

,Tipo,Nulos,% Nulos,Unicos
Columna,,,,
tipo_titulo,object,0,0.0,9
categoria,object,0,0.0,6
precio_eur,float64,0,0.0,9
es_abono,bool,0,0.0,2
bonificado,bool,0,0.0,2


In [164]:
df_tarifas.head(3)

,tipo_titulo,categoria,precio_eur,es_abono,bonificado
tarifa_id,,,,,
1,Ordinario,Adulto,1.50,False,False
2,Bono 10 Viajes,Adulto,0.97,True,False
3,Abono Mensual,Adulto,0.60,True,False


In [165]:
df_tarifas.describe()

,precio_eur
count,9.000000
mean,2.102222
std,3.336153
min,0.000000
25%,0.250000
50%,0.600000
75%,1.500000
max,10.000000


In [166]:
df_tarifas.describe(include='object')

,tipo_titulo,categoria
count,9,9
unique,9,6
top,Ordinario,Adulto
freq,1,3


## **Vehiculos**

In [167]:
df_vehiculos_metadatos = crear_tabla_metadatos(df_vehiculos)
df_vehiculos_metadatos

,Tipo,Nulos,% Nulos,Unicos
Columna,,,,
matricula,object,0,0.00,45
modelo,object,0,0.00,11
combustible,object,1,2.22,5
capacidad_sentados,int64,0,0.00,5
capacidad_total,int64,0,0.00,5
anno_fabricacion,int64,0,0.00,12
anno_incorporacion,int64,0,0.00,11
km_totales,int64,0,0.00,45
depot_id,int64,0,0.00,3


In [168]:
df_vehiculos.head(3)

,matricula,modelo,combustible,capacidad_sentados,capacidad_total,anno_fabricacion,anno_incorporacion,km_totales,depot_id,emisiones_co2_gkm,en_servicio
vehiculo_id,,,,,,,,,,,
1,6001 BUS,Mercedes-Benz Citaro,Diesel,88,128,2017,2018,586265,1,127,True
2,6002 BUS,Mercedes-Benz Citaro,Diesel,88,128,2018,2019,317095,1,112,True
3,6003 BUS,Iveco Urbanway,diesel,90,130,2018,2019,256606,2,133,True


In [169]:
df_vehiculos.describe()

,capacidad_sentados,capacidad_total,anno_fabricacion,anno_incorporacion,km_totales,depot_id,emisiones_co2_gkm
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,95.200000,135.200000,2020.822222,2020.133333,361823.533333,1.866667,83.466667
std,18.862662,18.862662,12.227556,2.792848,157356.850166,0.756787,54.024237
min,85.000000,125.000000,2014.000000,2015.000000,-500.000000,1.000000,0.000000
25%,88.000000,128.000000,2017.000000,2018.000000,274345.000000,1.000000,0.000000
50%,88.000000,128.000000,2019.000000,2020.000000,379164.000000,2.000000,109.000000
75%,90.000000,130.000000,2021.000000,2022.000000,495798.000000,2.000000,125.000000
max,145.000000,185.000000,2099.000000,2025.000000,596622.000000,3.000000,140.000000


In [170]:
df_vehiculos.describe(include='object')

,matricula,modelo,combustible
count,45,45,44
unique,45,11,4
top,6001 BUS,Mercedes-Benz Citaro,Diesel
freq,1,6,25
